In [ ]:
PAIR = "ru_ua"
SA, SB = {"ru_ua": ("ru", "ua"), "il_ps": ("ps", "il")}[PAIR]
LA, LB = SA.upper(), SB.upper()
RESULTS_DIR = {"ru_ua": "results", "il_ps": "results_il_ps"}[PAIR]

import os, json
import numpy as np


def load_all_examples_with_prefix(files_with_prefix):
    all_examples = {}
    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                new_id = f"{prefix}_{obj.get('id')}"
                obj["id"] = new_id
                all_examples[new_id] = obj
    return all_examples


def _find_latest_run(sd_path, prefix):
    runs = [d for d in os.listdir(sd_path) if d.startswith(prefix) and os.path.isdir(os.path.join(sd_path, d))]
    if not runs:
        return None
    runs.sort()
    fname = f"{SB}_results.jsonl" if prefix.startswith(SB) else f"{SA}_results.jsonl"
    fpath = os.path.join(sd_path, runs[-1], fname)
    return fpath if os.path.isfile(fpath) else None


def collect_latest_runs_all_subdirs(model_dir):
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"model_dir not found: {model_dir}")
    sb_files, sa_files = [], []
    for sd in sorted(d for d in os.listdir(model_dir) if os.path.isdir(os.path.join(model_dir, d))):
        sd_path = os.path.join(model_dir, sd)
        sb_path = _find_latest_run(sd_path, f"{SB}_run_")
        sa_path = _find_latest_run(sd_path, f"{SA}_run_")
        if sb_path and sa_path:
            sb_files.append((sd, sb_path))
            sa_files.append((sd, sa_path))
    return sb_files, sa_files


def scores_from_examples(examples, score_key):
    return {ex_id: float(obj[score_key]) for ex_id, obj in examples.items() if obj.get(score_key) is not None}


def _repo_root():
    p = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(p, RESULTS_DIR)):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            raise FileNotFoundError(f"no '{RESULTS_DIR}/' dir found above {os.getcwd()}")
        p = parent


In [ ]:
PRIMARY = ("llama3-8b", "BASE_JUDGE")
OTHERS = [
    ("falcon3-7b-instruct", "BASE_JUDGE"),
    ("gpt-5.1", "BASE_JUDGE"),
]
TREES = ["baseline", "control", "self_distance", "sdas_test"]
MODELS = None
TIES = "drop"
TIE_TOL = 0.0

if TIES not in ("drop", "label"):
    raise ValueError(f"TIES must be 'drop' or 'label' (got {TIES!r})")
SA_KEY = f"score_{SA}_mt_vs_{SA}_human"
SB_KEY = f"score_{SB}_mt_vs_{SB}_human"
RESULTS_ROOT = os.path.join(_repo_root(), RESULTS_DIR)


In [ ]:
def judge_dir(tree, judge):
    name, prompt = judge
    return os.path.join(RESULTS_ROOT, tree, f"{name}_judge", prompt)


def direction_labels(model_dir):
    sb_files, sa_files = collect_latest_runs_all_subdirs(model_dir)
    sa = scores_from_examples(load_all_examples_with_prefix(sa_files), SA_KEY)
    sb = scores_from_examples(load_all_examples_with_prefix(sb_files), SB_KEY)
    labels = {}
    for k in sa.keys() & sb.keys():
        diff = sb[k] - sa[k]
        labels[k] = 0 if abs(diff) <= TIE_TOL else (1 if diff > 0 else -1)
    return labels


def collect_decisions(judge):
    out = {}
    for tree in TREES:
        base = judge_dir(tree, judge)
        if not os.path.isdir(base):
            continue
        for model in sorted(os.listdir(base)):
            md = os.path.join(base, model)
            if not os.path.isdir(md) or (MODELS is not None and model not in MODELS):
                continue
            for k, v in direction_labels(md).items():
                out[(tree, model, k)] = v
    return out


def cohen_kappa(pairs, categories):
    n = len(pairs)
    if n == 0:
        return None
    idx = {c: i for i, c in enumerate(categories)}
    M = np.zeros((len(categories), len(categories)))
    for a, b in pairs:
        M[idx[a], idx[b]] += 1
    po = float(np.trace(M) / n)
    pe = float(np.sum(M.sum(axis=1) / n * M.sum(axis=0) / n))
    kappa = (po - pe) / (1.0 - pe) if (1.0 - pe) > 1e-12 else float("nan")
    return {"n": n, "agreement": po, "kappa": float(kappa), "confusion": M}


def compare(primary, other):
    common = sorted(primary.keys() & other.keys())
    pairs = [(primary[k], other[k]) for k in common]
    n_ties = sum(1 for a, b in pairs if a == 0 or b == 0)
    if TIES == "drop":
        kept = [(k, p) for k, p in zip(common, pairs) if p[0] != 0 and p[1] != 0]
        categories = [1, -1]
    else:
        kept = list(zip(common, pairs))
        categories = [1, 0, -1]
    return kept, n_ties, len(common), categories


primary = collect_decisions(PRIMARY)
print(f"{PAIR}: primary judge {PRIMARY[0]}_judge/{PRIMARY[1]}: {len(primary)} examples over {TREES}")
print(f"decision = sign of d({LB}) - d({LA}): +1 = closer to the {LA} reference, -1 = closer to the {LB} reference;  ties: {TIES} (tolerance {TIE_TOL})\n")

results = {}
for judge in OTHERS:
    other = collect_decisions(judge)
    kept, n_ties, n_common, categories = compare(primary, other)
    r = cohen_kappa([p for _, p in kept], categories)
    results[judge] = r
    name = f"{judge[0]}_judge/{judge[1]}"
    if r is None:
        print(f"{name}: no example scored by both judges ({len(other)} examples found for this judge)\n")
        continue
    print(f"{name}: kappa = {r['kappa']:.3f}   agreement = {r['agreement']:.3f}   n = {r['n']}"
          f"   (common examples {n_common}, with a tie {n_ties}{' - dropped' if TIES == 'drop' else ' - kept as a category'})")
    print(f"   confusion (rows = primary, cols = {judge[0]}; order {categories}):")
    for row in r["confusion"]:
        print("      " + "  ".join(f"{int(v):>6d}" for v in row))

    for level, key_of in (("condition", lambda k: k[0]), ("model", lambda k: k[1])):
        groups = {}
        for k, p in kept:
            groups.setdefault(key_of(k), []).append(p)
        print(f"   by {level}:")
        for g in sorted(groups):
            rg = cohen_kappa(groups[g], categories)
            print(f"      {g:<40} kappa = {rg['kappa']:+.3f}   agreement = {rg['agreement']:.3f}   n = {rg['n']}")
    print()
